# QuantJourney SDK - Multpl Market Valuation

This notebook demonstrates Multpl data:
- Shiller P/E Ratio (CAPE)
- S&P 500 Dividend Yield
- Market valuation metrics

**API:** https://api.quantjourney.cloud

## Run Output

![09_multpl_valuation](../plots/09_multpl_valuation_output_01.png)

**Prepared by QuantJourney.** Candidate notebook source is kept clean and unexecuted. Generated run artifacts are committed under `plots/` and indexed in `plots/manifest.json`.

In [ ]:
# Setup
import sys
sys.path.insert(0, '..')

from quantjourney.sdk import QuantJourneyAPI
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
pio.renderers.default = "png"

# API Key authentication
import os
API_KEY = os.environ.get("QJ_API_KEY", "qj_...")

qj = QuantJourneyAPI(api_key=API_KEY)
print("✓ Connected to QuantJourney API")


## 1. Shiller P/E Ratio (CAPE)

In [ ]:
# Fetch Shiller P/E (CAPE) - Cyclically Adjusted Price-to-Earnings
try:
    response = qj.multpl.get_shiller_pe_ratio()
    shiller_pe = response.get('value', response) if isinstance(response, dict) else response
    
    if shiller_pe:
        df_cape = pd.DataFrame(shiller_pe)
        if 'date' in df_cape.columns:
            df_cape['date'] = pd.to_datetime(df_cape['date'])
            df_cape = df_cape.sort_values('date')
            
            print(f"CAPE records: {len(df_cape)}")
            print(f"Date range: {df_cape['date'].min().date()} to {df_cape['date'].max().date()}")
            print(f"\nLatest readings:")
            print(df_cape.tail())
except Exception as e:
    print(f"Note: Multpl CAPE data - {e}")


In [ ]:
# Plot Shiller P/E
if 'df_cape' in dir() and len(df_cape) > 0:
    # Find value column
    value_col = [c for c in df_cape.columns if c not in ['date']][0]
    
    fig = go.Figure()
    
    fig.add_trace(go.Scatter(
        x=df_cape['date'],
        y=df_cape[value_col],
        name='Shiller P/E',
        line=dict(color='cyan')
    ))
    
    # Historical average
    avg = df_cape[value_col].mean()
    fig.add_hline(y=avg, line_dash='dash', line_color='orange',
                  annotation_text=f'Historical Avg: {avg:.1f}')
    
    # Overvalued/Undervalued zones
    fig.add_hline(y=25, line_dash='dot', line_color='yellow',
                  annotation_text='Overvalued (>25)')
    fig.add_hline(y=15, line_dash='dot', line_color='green',
                  annotation_text='Undervalued (<15)')
    
    fig.update_layout(
        title='Shiller P/E Ratio (CAPE) - Historical',
        yaxis_title='P/E Ratio',
        xaxis_title='Date',
        template='plotly_dark',
        height=500
    )
    fig.show()
    
    # Analysis
    current = df_cape[value_col].iloc[-1]
    print(f"\nShiller P/E Analysis:")
    print(f"  Current:    {current:.1f}")
    print(f"  Average:    {avg:.1f}")
    print(f"  Median:     {df_cape[value_col].median():.1f}")
    print(f"  Max:        {df_cape[value_col].max():.1f}")
    print(f"  Min:        {df_cape[value_col].min():.1f}")
    
    # Valuation assessment
    if current > avg * 1.5:
        status = "SIGNIFICANTLY OVERVALUED"
    elif current > avg * 1.2:
        status = "OVERVALUED"
    elif current > avg:
        status = "SLIGHTLY OVERVALUED"
    elif current > avg * 0.8:
        status = "FAIRLY VALUED"
    else:
        status = "UNDERVALUED"
    
    pct_vs_avg = (current / avg - 1) * 100
    print(f"  Vs Average: {pct_vs_avg:+.1f}%")
    print(f"  Valuation:  {status}")


## 2. S&P 500 Dividend Yield

In [ ]:
# Fetch S&P 500 Dividend Yield (TTM)
try:
    response = qj.multpl.get_dividend_yield_ttm()
    sp500_div = response.get('value', response) if isinstance(response, dict) else response
    
    if sp500_div:
        df_div = pd.DataFrame(sp500_div)
        # Find date column dynamically
        date_col = [c for c in df_div.columns if 'date' in c.lower()]
        if date_col:
            df_div['date'] = pd.to_datetime(df_div[date_col[0]])
            df_div = df_div.sort_values('date')
        
        print(f"Dividend Yield records: {len(df_div)}")
        print(f"Columns: {list(df_div.columns)}")
        if 'date' in df_div.columns:
            print(f"Date range: {df_div['date'].min().date()} to {df_div['date'].max().date()}")
        print(f"\nLatest data:")
        print(df_div.tail())
except Exception as e:
    print(f"Note: Dividend yield data - {e}")


In [ ]:
# Plot Dividend Yield
if 'df_div' in dir() and len(df_div) > 0:
    # Find the dividend yield value column
    value_col = 'dividend_yield' if 'dividend_yield' in df_div.columns else [c for c in df_div.columns if c not in ['date', 'frequency', 'series_name']][0]
    
    # Convert string values like "1.71%" to numeric
    df_div['yield_numeric'] = df_div[value_col].astype(str).str.replace('†', '').str.replace('%', '').astype(float)
    
    # Last 20 years
    df_div_recent = df_div[df_div['date'] >= '2005-01-01'].copy()
    
    if len(df_div_recent) > 0:
        fig = go.Figure()
        fig.add_trace(go.Scatter(
            x=df_div_recent['date'],
            y=df_div_recent['yield_numeric'],
            fill='tozeroy',
            name='Dividend Yield',
            line=dict(color='green')
        ))
        
        avg = df_div_recent['yield_numeric'].mean()
        fig.add_hline(y=avg, line_dash='dash', line_color='orange',
                      annotation_text=f'Avg: {avg:.2f}%')
        
        fig.update_layout(
            title='S&P 500 Dividend Yield (%)',
            yaxis_title='Yield (%)',
            xaxis_title='Date',
            template='plotly_dark',
            height=450
        )
        fig.show()
        
        print(f"\nDividend Yield Analysis:")
        print(f"  Current:  {df_div_recent['yield_numeric'].iloc[-1]:.2f}%")
        print(f"  Average:  {avg:.2f}%")
        print(f"  Max:      {df_div_recent['yield_numeric'].max():.2f}%")
        print(f"  Min:      {df_div_recent['yield_numeric'].min():.2f}%")
    else:
        print("No data for the last 20 years")


## 3. Combined Valuation View

In [ ]:
# Summary dashboard
print("="*60)
print("MARKET VALUATION SUMMARY")
print("="*60)

metrics = []

if 'df_cape' in dir() and len(df_cape) > 0:
    # Find numeric value column for CAPE
    cape_col = 'shiller_pe' if 'shiller_pe' in df_cape.columns else [c for c in df_cape.columns if c not in ['date', 'frequency', 'series_name']][0]
    current = df_cape[cape_col].iloc[-1]
    avg = df_cape[cape_col].mean()
    metrics.append(("Shiller P/E (CAPE)", f"{current:.1f}", f"{(current/avg-1)*100:+.1f}% vs avg"))

if 'df_div' in dir() and len(df_div) > 0:
    # Use yield_numeric if available (already converted from string)
    if 'yield_numeric' in df_div.columns:
        current = df_div['yield_numeric'].iloc[-1]
        avg = df_div['yield_numeric'].mean()
    else:
        value_col = [c for c in df_div.columns if c not in ['date', 'frequency', 'series_name']][0]
        current = float(str(df_div[value_col].iloc[-1]).replace('†', '').replace('%', ''))
        avg = df_div[value_col].astype(str).str.replace('†', '').str.replace('%', '').astype(float).mean()
    metrics.append(("S&P 500 Div Yield", f"{current:.2f}%", f"{(current/avg-1)*100:+.1f}% vs avg"))

if metrics:
    print(f"\n{'Metric':<25} {'Current':<12} {'vs Average'}")
    print("-" * 55)
    for name, current, vs_avg in metrics:
        print(f"{name:<25} {current:<12} {vs_avg}")
else:
    print("\nNo valuation data available")

print("\n" + "="*60)


## Summary

Multpl valuation metrics covered:
- **Shiller P/E (CAPE)**: Cyclically adjusted P/E ratio
- **S&P 500 Dividend Yield**: Market income return

### Interpretation:
- **High CAPE (>25)**: Market potentially overvalued, lower expected returns
- **Low CAPE (<15)**: Market potentially undervalued, higher expected returns
- **High Dividend Yield**: Attractive income, potential value
- **Low Dividend Yield**: Growth expectations priced in